In [1]:
import robotic as ry
import random
import numpy as np
import os

from WayTu_RAI.GenerateEnvironment import GenerateEnvironment, PlacementError
import WayTu_RAI.model_utils as mutils



ry.params_add({'physx/motorKp': 10000., 
               'physx/motorK  d': 1000., 
               'physx/angularDamping': 10., 
               'physx/defaultFriction': 1000.})

ry.params_add({'botsim/engine': 'physx'}) #makes a big difference!
ry.params_add({'physx/multibody': True}) #makes a big difference!
ry.params_print()


test_seeds = [1567]

def goal_position_calculation(C, task): 
    if task == "minigolf":
        goal_position = (C.getFrame("left-area").getPosition() + C.getFrame("right-area").getPosition())/2
    elif task ==  "lifting": 
        goal_position = C.getFrame("lifting-obj").getPosition()
        goal_position[2] += 0.1
    elif task == "hammering": 
        left_center = C.getFrame("left-area").getPosition()
        right_center = C.getFrame("right-area").getPosition()
        goal_position = (left_center + right_center) / 2
    # C.addFrame("target-obj-goal")\
    #     .setShape(ry.ST.marker, [.1])\
    #     .setPosition(goal_position)\

    return goal_position

def just_motion_optimizer_test(parameters):
    trial = 0
    success = 0
    while trial < parameters["num-trials"]:
        try:
            if parameters["use-test-seed"]:
                seed = test_seeds[trial]
                random.seed(seed)
                np.random.seed(seed)
            if parameters["use-test-set"] == False: 
                environment = GenerateEnvironment(parameters)
                environment.generate_environment()
                environment.C.view()
            else: 
                g_path = os.path.join(parameters["dataset-dir"], f"data_{trial}_env.g")
                environment = GenerateEnvironment(parameters)
                environment.C.addFile(g_path)
                environment.set_tool_objs(parameters["possible-tools"])
                environment.C.view()
                # environment.C.ensure_proper_proxies()
            
            print(environment.tool_objs)
            # raise Exception
            
            tools = [f.name for f in environment.C.frames() if any(k in f.name for k in parameters["possible-tools"])and "platform" not in f.name and "obj" not in f.name]
            print("tools: ", tools)
            selected_part = random.choice(tools)
            best_tool, part_name = selected_part.rsplit("-", 1)
            other_tools = [item for item in environment.tool_objs if item != best_tool]
            mutils.reparent_tool(environment.C, best_tool, selected_part, other_tools)

            print("After reparent tools.")


            target_object = parameters["task"] + "-obj"  
            # goal_position = "target-obj-goal"
            if part_name == "base":
                interaction_part = best_tool + "-head"
            else:
                interaction_part = best_tool + "-base"

            goal_position = goal_position_calculation(environment.C, parameters["task"])
            # q0 = environment.C.getJointState()
            print("Before Starting the KOMO.")
            success_in_env = False
            for attempt in range(parameters.get("num-attempts", 10)):
                print(f"  Attempt {attempt+1}/{parameters.get('num-attempts', 10)}")

                # Randomize starting joint configuration a bit
                q0 = environment.C.getJointState()
                q0 += 0.05 * (2 * np.random.rand(len(q0)) - 1)  # small noise
                environment.C.setJointState(q0)

                # KOMO setup
                komo = ry.KOMO()
                komo.setConfig(environment.C, True)
                komo.setTiming(3, 10, 3.0, 2)

                # Motion constraints
                komo.addControlObjective([], order=2, scale=1.0)
                komo.addObjective([], ry.FS.accumulatedCollisions, [], ry.OT.eq)
                komo.addObjective([], ry.FS.jointLimits, [], ry.OT.ineq)
                komo.addObjective([], ry.FS.jointState, [], ry.OT.sos, [1e-1], q0)

                # Phase 1: Grasp
                grasp_time = 1.0
                komo.addObjective([grasp_time],
                    ry.FS.distance, ["l_gripper", selected_part],
                    ry.OT.eq, [0.0])
                komo.addObjective([grasp_time],
                    ry.FS.positionDiff, ["l_gripper", selected_part],
                    ry.OT.eq, [0.001])
                komo.addObjective([grasp_time],
                    ry.FS.scalarProductXY, ["l_gripper", selected_part],
                    ry.OT.eq, [1.0])

                # Phase 2: Interaction
                interaction_time = 2.0
                komo.addObjective([interaction_time],
                    ry.FS.positionDiff, [interaction_part, target_object],
                    ry.OT.eq, [0.001])

                # Phase 3: Goal
                total_time = 3.0
                komo.addObjective([total_time],
                    ry.FS.position, [target_object],
                    ry.OT.eq, goal_position)

                # Solve KOMO
                solver = ry.NLP_Solver()
                solver.setProblem(komo.nlp())
                ret = solver.solve()

                report = komo.report()
                print(report)

                if ret.feasible:
                    print(f"    [OK] Feasible solution found on attempt {attempt+1}")
                    success_in_env = True
                    break  # no need to try more attempts for this env
                else:
                    print(f"    [FAIL] No feasible solution on attempt {attempt+1}")

            if success_in_env:
                print(f"[OK] Environment {trial} solved by KOMO.")
                success += 1
            else:
                print(f"[FAIL] Environment {trial} failed in all attempts.")

            trial += 1

        except PlacementError as e:
            print(f"[!] Skipping trial due to placement error: {e}")
            continue

    print(f"Success Rate: {success}/{parameters['num-trials']}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
physx/motorKp: 10000,
physx/motorK  d: 1000,
physx/angularDamping: 10,
physx/defaultFriction: 1000,
botsim/engine: physx,
physx/multibody

In [2]:
parameters = { 
    "mode" : "test", 
    "num-tools" : 3,
    "num-obj-points": 512,
    "label-list-all" : ["lifting-platform", "minigolf-platform", "hammering-platform",
                  "hammer", "spatula", "L-ruler", "screwdriver", "ball", "book", "thin-stick", "ring"],
    "model-name" : "waytu_unified_model_v11_trial_hammering_best.pth",
    "feature-extractor-path" : "small-pointner-encoder-distractor_best.pth",
    "feature-size" : 128,
    "num-trials" : 1, 
    "tool-type" : "primitive",
    "task" : "hammering", 
    "area-middle": {
        "min" : [-0.35 , 0.15, 0.060],
        "max" : [ 0.35 , 0.45, 0.065]
        },
    "area-negative" : {
        "min": [-0.50 , 0.15, 0.050],
        "max": [ -0.35 , 0.35, 0.060],
        },
    "area-positive" : {
        "min": [0.40 , 0.15, 0.050],
        "max": [ 0.50 , 0.35, 0.060],
        },
    "cameras" : ["camera1", "camera2", "camera3" ],
    "possible-tools" : ["hammer", "spatula", "L-ruler"],
    "use-test-set" : False,
    "use-test-seed" : False,
    "dataset-dir" : "minigolf-test-dataset",
    "num-attempts" : 1,
    
 
}

just_motion_optimizer_test(parameters)

{'name': 'table', 'ID': 1, 'rel': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0], 'shape': 'ssBox', 'size': [2.5, 2.5, 0.1, 0.02], 'color': [0.3, 0.3, 0.3], 'contact': 1, 'logical': {}, 'friction': 0.1, 'X': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0]}
table threshold:  0.655
camera height:  [0.   0.9  1.22]
{'min': array([0.25279772, 0.26166524, 0.6158102 ]), 'max': array([0.39279772, 0.58166524, 0.7158102 ]), 'center': array([0.32279772, 0.42166524, 0.6658102 ]), 'quaternion': array([ 0.94604473,  0.        ,  0.        , -0.32403605]), 'rotation_z': -37.81436962515161}
(3435, 3)
(3435, 3)
[0, 0, 255]
obj name: hammer, label: 3
{'min': array([-0.09771617,  0.12660059,  0.62398756]), 'max': array([0.20228383, 0.44660059, 0.70398756]), 'center': array([0.05228383, 0.28660059, 0.66398756]), 'quaternion': array([ 0.93493657,  0.        ,  0.        , -0.35481489]), 'rotation_z': -41.56419911057056}
{'min': array([-0.10783215,  0.25327103,  0.62085143]), 'max': array([0.19216785, 0.57327103, 0.70085143]), '